In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **00_config**

In [2]:
# ============================================================
# 00_config
# ViT-B/16 + Patch-level Background Suppression
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/MLP/Project5")
DATA_ROOT = PROJECT_ROOT / "dataset_expand"
OUTPUT_ROOT = PROJECT_ROOT / "final_outputs"

CONFIG = {
    "experiment_name": "vit_b16_bg_suppression_patch_lambda0001_expand_10ep",
    "model_name": "vit_b16",
    "method_name": "Patch-level Background Suppression",

    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "output_dir": str(OUTPUT_ROOT / "vit_b16_bg_suppression_patch_lambda0001_expand_10ep"),

    "train_dir": str(DATA_ROOT / "train"),
    "val_dir": str(DATA_ROOT / "val"),
    "test_original_dir": str(DATA_ROOT / "test_original"),
    "test_jpeg_dir": str(DATA_ROOT / "test_jpeg_q30_controlled"),

    "image_size": 224,
    "patch_grid": 14,
    "batch_size": 16,
    "num_workers": 2,
    "num_epochs": 10,

    "learning_rate": 1e-5,
    "weight_decay": 1e-4,
    "lambda_bg": 0.001,

    "seed": 42,
    "positive_class_name": "fake",

    "mask_cache_dir": str(PROJECT_ROOT / "mask_cache" / "deeplabv3_patch_masks_224_train"),
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["mask_cache_dir"], exist_ok=True)

with open(os.path.join(CONFIG["output_dir"], "config.json"), "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{
  "experiment_name": "vit_b16_bg_suppression_patch_lambda0001_expand_10ep",
  "model_name": "vit_b16",
  "method_name": "Patch-level Background Suppression",
  "project_root": "/content/drive/MyDrive/MLP/Project5",
  "data_root": "/content/drive/MyDrive/MLP/Project5/dataset_expand",
  "output_root": "/content/drive/MyDrive/MLP/Project5/final_outputs",
  "output_dir": "/content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_bg_suppression_patch_lambda0001_expand_10ep",
  "train_dir": "/content/drive/MyDrive/MLP/Project5/dataset_expand/train",
  "val_dir": "/content/drive/MyDrive/MLP/Project5/dataset_expand/val",
  "test_original_dir": "/content/drive/MyDrive/MLP/Project5/dataset_expand/test_original",
  "test_jpeg_dir": "/content/drive/MyDrive/MLP/Project5/dataset_expand/test_jpeg_q30_controlled",
  "image_size": 224,
  "patch_grid": 14,
  "batch_size": 16

# **01_imports_and_seed**

In [3]:
# ============================================================
# 01_imports_and_seed
# ============================================================

import os
import time
import json
import random
import math
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from tqdm.auto import tqdm

try:
    import timm
except ImportError:
    !pip -q install timm
    import timm


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


# **02_dataset_basic**

In [4]:
# ============================================================
# 02_dataset_basic
# Basic datasets and transforms
# ============================================================

def rgb_loader(path):
    with open(path, "rb") as f:
        img = Image.open(f)
        return img.convert("RGB")


image_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_base = ImageFolder(
    CONFIG["train_dir"],
    transform=image_transform,
    loader=rgb_loader
)

val_base = ImageFolder(
    CONFIG["val_dir"],
    transform=image_transform,
    loader=rgb_loader
)

class_names = train_base.classes
class_to_idx = train_base.class_to_idx

print("class_names:", class_names)
print("class_to_idx:", class_to_idx)
print("train size:", len(train_base))
print("val size:", len(val_base))

positive_class_idx = class_to_idx[CONFIG["positive_class_name"]]
print("positive_class_idx:", positive_class_idx)

class_names: ['fake', 'real']
class_to_idx: {'fake': 0, 'real': 1}
train size: 9996
val size: 2004
positive_class_idx: 0


# **03_precompute_patch_background_masks**
这个 cell 会先用 DeepLabV3 给 train images 生成 14×14 patch background mask。
如果之前已经生成过，会直接跳过已有 mask。

In [6]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [7]:
# ============================================================
# 03_precompute_patch_background_masks_SAFE
# Generate DeepLabV3 patch-level background masks for train set
# Safe version: handles truncated/corrupted images
# ============================================================

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

mask_cache_dir = Path(CONFIG["mask_cache_dir"])
mask_cache_dir.mkdir(parents=True, exist_ok=True)

seg_weights = DeepLabV3_ResNet50_Weights.DEFAULT
seg_model = deeplabv3_resnet50(weights=seg_weights).to(device)
seg_model.eval()

seg_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def safe_mask_filename(path):
    rel = str(Path(path).relative_to(CONFIG["train_dir"]))
    rel = rel.replace("/", "__").replace("\\", "__")
    return rel + ".npy"


class RawPathDatasetSafe(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        try:
            img = rgb_loader(path)
            x = seg_transform(img)
            valid = True
        except Exception as e:
            # Bad / truncated image fallback
            x = torch.zeros(3, CONFIG["image_size"], CONFIG["image_size"])
            valid = False

        return x, path, label, valid


raw_train_ds = RawPathDatasetSafe(train_base.samples)

# Important: num_workers=0 makes debugging safer for corrupted images
raw_train_loader = DataLoader(
    raw_train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

existing_count = len(list(mask_cache_dir.glob("*.npy")))
print("Existing cached masks:", existing_count)
print("Target train masks:", len(raw_train_ds))

start_time = time.time()
generated = 0
skipped = 0
bad_images = []

with torch.no_grad():
    for images, paths, labels, valid_flags in tqdm(raw_train_loader, desc="Generating patch BG masks SAFE"):

        dst_paths = [mask_cache_dir / safe_mask_filename(p) for p in paths]

        # Skip already cached masks
        need_indices = [i for i, p in enumerate(dst_paths) if not p.exists()]

        if len(need_indices) == 0:
            skipped += len(paths)
            continue

        need_images = images[need_indices]
        need_paths = [paths[i] for i in need_indices]
        need_dst_paths = [dst_paths[i] for i in need_indices]
        need_valid_flags = [bool(valid_flags[i]) for i in need_indices]

        # Save zero masks directly for invalid images
        valid_indices_local = []
        for local_i, is_valid in enumerate(need_valid_flags):
            if not is_valid:
                zero_mask = np.zeros(CONFIG["patch_grid"] * CONFIG["patch_grid"], dtype=np.float32)
                np.save(need_dst_paths[local_i], zero_mask)
                bad_images.append(need_paths[local_i])
                generated += 1
            else:
                valid_indices_local.append(local_i)

        if len(valid_indices_local) == 0:
            continue

        valid_images = need_images[valid_indices_local].to(device, non_blocking=True)
        valid_dst_paths = [need_dst_paths[i] for i in valid_indices_local]

        out = seg_model(valid_images)["out"]  # [B, 21, H, W]
        pred = out.argmax(dim=1)             # [B, H, W]

        # DeepLab background class = 0
        fg = (pred != 0).float().unsqueeze(1)  # [B,1,224,224]

        # Downsample to ViT patch grid 14x14
        fg_patch_ratio = F.adaptive_avg_pool2d(
            fg,
            output_size=(CONFIG["patch_grid"], CONFIG["patch_grid"])
        ).squeeze(1)  # [B,14,14]

        # background patch = foreground ratio < 0.5
        bg_patch = (fg_patch_ratio < 0.5).float()  # [B,14,14]

        # If DeepLab detects no foreground, do not apply BG suppression for this image
        fg_sum = fg_patch_ratio.flatten(1).sum(dim=1)
        bg_patch[fg_sum == 0] = 0.0

        bg_patch_flat = bg_patch.flatten(1).cpu().numpy().astype(np.float32)  # [B,196]

        for arr, dst in zip(bg_patch_flat, valid_dst_paths):
            np.save(dst, arr)
            generated += 1

elapsed_min = (time.time() - start_time) / 60
final_count = len(list(mask_cache_dir.glob("*.npy")))

print("Generated new masks:", generated)
print("Skipped existing masks:", skipped)
print("Final cached masks:", final_count)
print("Target train masks:", len(raw_train_ds))
print("Bad/truncated images handled:", len(bad_images))
print("Elapsed min:", elapsed_min)
print("Mask cache dir:", mask_cache_dir)

if len(bad_images) > 0:
    bad_log_path = Path(CONFIG["output_dir"]) / "bad_images_during_mask_generation.txt"
    with open(bad_log_path, "w", encoding="utf-8") as f:
        for p in bad_images:
            f.write(str(p) + "\n")
    print("Saved bad image log:", bad_log_path)
    print("First bad images:")
    for p in bad_images[:5]:
        print(p)

Existing cached masks: 5664
Target train masks: 9996


Generating patch BG masks SAFE:   0%|          | 0/625 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (98058240 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (90671520 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96012000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107184040 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Generated new masks: 4332
Skipped existing masks: 5664
Final cached masks: 9996
Target train masks: 9996
Bad/truncated images handled: 0
Elapsed min: 45.675241891543074
Mask cache dir: /content/drive/MyDrive/MLP/Project5/mask_cache/deeplabv3_patch_masks_224_train


# **04_dataset_with_patch_masks**

In [8]:
# ============================================================
# 04_dataset_with_patch_masks
# Dataset returns image, label, and patch-level background mask
# ============================================================

class ImageFolderWithPatchMask(Dataset):
    def __init__(self, imagefolder, mask_cache_dir=None, train_dir=None, return_mask=True):
        self.imagefolder = imagefolder
        self.samples = imagefolder.samples
        self.classes = imagefolder.classes
        self.class_to_idx = imagefolder.class_to_idx
        self.mask_cache_dir = Path(mask_cache_dir) if mask_cache_dir is not None else None
        self.train_dir = train_dir
        self.return_mask = return_mask

    def __len__(self):
        return len(self.imagefolder)

    def _mask_path(self, image_path):
        rel = str(Path(image_path).relative_to(self.train_dir))
        rel = rel.replace("/", "__").replace("\\", "__")
        return self.mask_cache_dir / (rel + ".npy")

    def __getitem__(self, idx):
        image, label = self.imagefolder[idx]
        image_path, _ = self.samples[idx]

        if self.return_mask:
            mp = self._mask_path(image_path)
            if mp.exists():
                bg_mask = np.load(mp).astype(np.float32)
            else:
                # fallback: no background suppression for this image
                bg_mask = np.zeros(CONFIG["patch_grid"] * CONFIG["patch_grid"], dtype=np.float32)
            bg_mask = torch.from_numpy(bg_mask)
            return image, label, bg_mask, image_path

        return image, label, image_path


train_ds = ImageFolderWithPatchMask(
    train_base,
    mask_cache_dir=CONFIG["mask_cache_dir"],
    train_dir=CONFIG["train_dir"],
    return_mask=True
)

val_ds = ImageFolderWithPatchMask(
    val_base,
    return_mask=False
)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)

batch = next(iter(train_loader))
images, labels, bg_masks, paths = batch

print("images:", images.shape)
print("labels:", labels.shape)
print("bg_masks:", bg_masks.shape)
print("bg mask mean:", bg_masks.float().mean().item())
print("example path:", paths[0])

images: torch.Size([16, 3, 224, 224])
labels: torch.Size([16])
bg_masks: torch.Size([16, 196])
bg mask mean: 0.4285714328289032
example path: /content/drive/MyDrive/MLP/Project5/dataset_expand/train/fake/4859.jpg


# **05_model_vit_bg_suppression**

In [9]:
# ============================================================
# 05_model_vit_bg_suppression
# ViT-B/16 with CLS classification + patch background suppression
# ============================================================

class ViTBackgroundSuppressionClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=0
        )
        self.embed_dim = self.backbone.num_features
        self.classifier = nn.Linear(self.embed_dim, num_classes)

    def forward(self, x, return_patch_tokens=False):
        features = self.backbone.forward_features(x)

        if features.ndim != 3:
            raise ValueError(f"Expected ViT features [B, tokens, D], got {features.shape}")

        cls_token = features[:, 0, :]       # [B,D]
        patch_tokens = features[:, 1:, :]   # [B,196,D]

        logits = self.classifier(cls_token)

        if return_patch_tokens:
            return logits, patch_tokens

        return logits


def patch_background_suppression_loss(patch_tokens, bg_masks, eps=1e-6):
    """
    patch_tokens: [B, N, D]
    bg_masks: [B, N], 1 = background patch, 0 = non-background / ignored
    """
    B, N, D = patch_tokens.shape

    bg_masks = bg_masks.to(patch_tokens.device).float()

    # token activation magnitude, normalized by sqrt(D)
    token_norm = torch.norm(patch_tokens, p=2, dim=-1) / math.sqrt(D)  # [B,N]

    bg_loss = (token_norm * bg_masks).sum() / (bg_masks.sum() + eps)

    return bg_loss


model = ViTBackgroundSuppressionClassifier(
    num_classes=len(class_names)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

print(model.__class__.__name__)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

ViTBackgroundSuppressionClassifier


# **06_train_eval_functions**

In [10]:
# ============================================================
# 06_train_eval_functions
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, device, lambda_bg):
    model.train()

    all_preds = []
    all_labels = []

    total_loss = 0.0
    total_ce_loss = 0.0
    total_bg_loss = 0.0
    total_samples = 0

    pbar = tqdm(loader, desc="Train", leave=False)

    for images, labels, bg_masks, paths in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        bg_masks = bg_masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits, patch_tokens = model(images, return_patch_tokens=True)

        ce_loss = criterion(logits, labels)
        bg_loss = patch_background_suppression_loss(patch_tokens, bg_masks)
        loss = ce_loss + lambda_bg * bg_loss

        loss.backward()
        optimizer.step()

        preds = logits.argmax(dim=1)

        batch_size = labels.size(0)
        total_samples += batch_size

        total_loss += loss.item() * batch_size
        total_ce_loss += ce_loss.item() * batch_size
        total_bg_loss += bg_loss.item() * batch_size

        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "ce": f"{ce_loss.item():.4f}",
            "bg": f"{bg_loss.item():.4f}",
        })

    avg_loss = total_loss / total_samples
    avg_ce_loss = total_ce_loss / total_samples
    avg_bg_loss = total_bg_loss / total_samples

    acc = accuracy_score(all_labels, all_preds)

    return {
        "train_loss": avg_loss,
        "train_ce_loss": avg_ce_loss,
        "train_bg_loss": avg_bg_loss,
        "train_acc": acc,
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device, positive_class_idx):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    total_loss = 0.0
    total_samples = 0

    pbar = tqdm(loader, desc="Eval", leave=False)

    for images, labels, paths in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images, return_patch_tokens=False)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)

        batch_size = labels.size(0)
        total_samples += batch_size
        total_loss += loss.item() * batch_size

        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_probs.extend(probs.detach().cpu().numpy().tolist())

    avg_loss = total_loss / total_samples

    labels_np = np.array(all_labels)
    preds_np = np.array(all_preds)

    acc = accuracy_score(labels_np, preds_np)
    precision_fake = precision_score(labels_np, preds_np, pos_label=positive_class_idx, zero_division=0)
    recall_fake = recall_score(labels_np, preds_np, pos_label=positive_class_idx, zero_division=0)
    f1_fake = f1_score(labels_np, preds_np, pos_label=positive_class_idx, zero_division=0)
    cm = confusion_matrix(labels_np, preds_np)

    return {
        "val_loss": avg_loss,
        "val_acc": acc,
        "val_precision_fake": precision_fake,
        "val_recall_fake": recall_fake,
        "val_f1_fake": f1_fake,
        "val_confusion_matrix": cm.tolist(),
    }

# **07_train_and_save_best_checkpoint**

In [11]:
# ============================================================
# 07_train_and_save_best_checkpoint
# ============================================================

output_dir = Path(CONFIG["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

best_ckpt_path = output_dir / "best_checkpoint.pt"
last_ckpt_path = output_dir / "last_checkpoint.pt"
log_path = output_dir / "training_log.csv"

start_epoch = 1
best_val_f1 = -1.0
best_epoch = -1

training_logs = []

# Resume if last checkpoint exists
if last_ckpt_path.exists():
    print("Found last checkpoint. Resuming from:", last_ckpt_path)
    ckpt = torch.load(last_ckpt_path, map_location=device, weights_only=False)

    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])

    start_epoch = ckpt["epoch"] + 1
    best_val_f1 = ckpt.get("best_val_f1", -1.0)
    best_epoch = ckpt.get("best_epoch", -1)

    if log_path.exists():
        training_logs = pd.read_csv(log_path).to_dict("records")

    print("Resume start_epoch:", start_epoch)
    print("Current best_val_f1:", best_val_f1)
    print("Current best_epoch:", best_epoch)

total_start = time.time()

for epoch in range(start_epoch, CONFIG["num_epochs"] + 1):
    epoch_start = time.time()

    print(f"\n========== Epoch {epoch}/{CONFIG['num_epochs']} ==========")

    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        lambda_bg=CONFIG["lambda_bg"]
    )

    val_metrics = evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        positive_class_idx=positive_class_idx
    )

    epoch_time_min = (time.time() - epoch_start) / 60

    log_row = {
        "epoch": epoch,
        **train_metrics,
        **val_metrics,
        "epoch_time_min": epoch_time_min,
        "lambda_bg": CONFIG["lambda_bg"],
    }

    training_logs.append(log_row)

    pd.DataFrame(training_logs).to_csv(log_path, index=False, encoding="utf-8-sig")

    print(
        f"Epoch {epoch} | "
        f"train_loss={train_metrics['train_loss']:.4f} | "
        f"ce={train_metrics['train_ce_loss']:.4f} | "
        f"bg={train_metrics['train_bg_loss']:.4f} | "
        f"train_acc={train_metrics['train_acc']:.4f} | "
        f"val_loss={val_metrics['val_loss']:.4f} | "
        f"val_acc={val_metrics['val_acc']:.4f} | "
        f"val_f1_fake={val_metrics['val_f1_fake']:.4f} | "
        f"time={epoch_time_min:.2f} min"
    )

    current_val_f1 = val_metrics["val_f1_fake"]

    # Save best checkpoint
    if current_val_f1 > best_val_f1:
        best_val_f1 = current_val_f1
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_val_f1": best_val_f1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": CONFIG,
                "class_names": class_names,
                "class_to_idx": class_to_idx,
                "positive_class_idx": positive_class_idx,
            },
            best_ckpt_path
        )

        print(f"Saved best checkpoint: epoch={epoch}, best_val_f1={best_val_f1:.6f}")

    # Save last checkpoint
    torch.save(
        {
            "epoch": epoch,
            "best_epoch": best_epoch,
            "best_val_f1": best_val_f1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": CONFIG,
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "positive_class_idx": positive_class_idx,
        },
        last_ckpt_path
    )

total_time_min = (time.time() - total_start) / 60

print("\nTraining finished.")
print("Best epoch:", best_epoch)
print("Best val F1:", best_val_f1)
print("Total training time min:", total_time_min)
print("Best checkpoint:", best_ckpt_path)
print("Last checkpoint:", last_ckpt_path)
print("Training log:", log_path)


========== Epoch 1/10 ==========


Train:   0%|          | 0/625 [00:00<?, ?it/s]

Eval:   0%|          | 0/126 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (161087488 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Epoch 1 | train_loss=0.2754 | ce=0.2731 | bg=2.2556 | train_acc=0.8835 | val_loss=0.1561 | val_acc=0.9431 | val_f1_fake=0.9443 | time=25.79 min
Saved best checkpoint: epoch=1, best_val_f1=0.944282

========== Epoch 2/10 ==========


...

Train:   0%|          | 0/625 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96012000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (107184040 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (90671520 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-p

Eval:   0%|          | 0/126 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (161087488 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (96000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Epoch 10 | train_loss=0.0170 | ce=0.0144 | bg=2.6182 | train_acc=0.9949 | val_loss=0.2498 | val_acc=0.9416 | val_f1_fake=0.9408 | time=18.96 min

Training finished.
Best epoch: 5
Best val F1: 0.948768472906404
Total training time min: 203.01152755419415
Best checkpoint: /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_bg_suppression_patch_lambda0001_expand_10ep/best_checkpoint.pt
Last checkpoint: /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_bg_suppression_patch_lambda0001_expand_10ep/last_checkpoint.pt
Training log: /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_bg_suppression_patch_lambda0001_expand_10ep/training_log.csv


# **08_test_original_and_jpeg**

In [12]:
# ============================================================
# 08_test_original_and_jpeg
# Evaluate best checkpoint on original test and JPEG q30 test
# Save metrics + predictions CSV
# ============================================================

import os
import json
import time
from pathlib import Path

import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


output_dir = Path(CONFIG["output_dir"])
eval_dir = output_dir / "eval_jpeg_q30_controlled"
eval_dir.mkdir(parents=True, exist_ok=True)

best_ckpt_path = output_dir / "best_checkpoint.pt"

print("Loading best checkpoint:", best_ckpt_path)

ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)

model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

best_epoch = ckpt.get("best_epoch", ckpt.get("epoch", None))
best_val_f1 = ckpt.get("best_val_f1", None)

print("Best epoch:", best_epoch)
print("Best val F1:", best_val_f1)


# ------------------------------------------------------------
# Dataset that returns image, label, and image path
# ------------------------------------------------------------

class ImageFolderWithPath(Dataset):
    def __init__(self, root_dir, transform, loader=rgb_loader):
        self.dataset = ImageFolder(
            root=root_dir,
            transform=transform,
            loader=loader
        )
        self.samples = self.dataset.samples
        self.classes = self.dataset.classes
        self.class_to_idx = self.dataset.class_to_idx

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        image_path, _ = self.samples[idx]
        return image, label, image_path


test_original_ds = ImageFolderWithPath(
    CONFIG["test_original_dir"],
    transform=image_transform,
    loader=rgb_loader
)

test_jpeg_ds = ImageFolderWithPath(
    CONFIG["test_jpeg_dir"],
    transform=image_transform,
    loader=rgb_loader
)

test_original_loader = DataLoader(
    test_original_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_jpeg_loader = DataLoader(
    test_jpeg_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Original test size:", len(test_original_ds))
print("JPEG test size:", len(test_jpeg_ds))
print("Original class_to_idx:", test_original_ds.class_to_idx)
print("JPEG class_to_idx:", test_jpeg_ds.class_to_idx)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def get_image_id(path):
    """
    Make original/JPEG pairable image_id.
    Example:
    xxx.jpg      -> xxx
    xxx_JPEG.jpg -> xxx
    """
    stem = Path(path).stem

    if stem.endswith("_JPEG"):
        stem = stem[:-5]

    return stem


@torch.no_grad()
def evaluate_and_save_predictions(
    model,
    loader,
    criterion,
    device,
    class_names,
    positive_class_idx,
    save_csv_path,
    split_name
):
    model.eval()

    all_rows = []
    all_labels = []
    all_preds = []

    total_loss = 0.0
    total_samples = 0

    start_time = time.time()

    pbar = tqdm(loader, desc=f"Eval {split_name}", leave=False)

    for images, labels, image_paths in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images, return_patch_tokens=False)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)

        batch_size = labels.size(0)
        total_samples += batch_size
        total_loss += loss.item() * batch_size

        labels_cpu = labels.detach().cpu().numpy()
        preds_cpu = preds.detach().cpu().numpy()
        probs_cpu = probs.detach().cpu().numpy()

        all_labels.extend(labels_cpu.tolist())
        all_preds.extend(preds_cpu.tolist())

        for i in range(batch_size):
            image_path = image_paths[i]
            filename = Path(image_path).name

            true_idx = int(labels_cpu[i])
            pred_idx = int(preds_cpu[i])

            row = {
                "split": split_name,
                "image_path": image_path,
                "filename": filename,
                "image_id": get_image_id(image_path),
                "file_extension": Path(filename).suffix.lower(),

                "true_label_idx": true_idx,
                "true_label_name": class_names[true_idx],
                "pred_label_idx": pred_idx,
                "pred_label_name": class_names[pred_idx],
                "correct": int(true_idx == pred_idx),

                "prob_fake": float(probs_cpu[i][class_to_idx["fake"]]),
                "prob_real": float(probs_cpu[i][class_to_idx["real"]]),
            }

            # Also save class probabilities in generic format
            for c_idx, c_name in enumerate(class_names):
                row[f"prob_{c_name}"] = float(probs_cpu[i][c_idx])

            all_rows.append(row)

    elapsed_sec = time.time() - start_time
    elapsed_min = elapsed_sec / 60

    labels_np = np.array(all_labels)
    preds_np = np.array(all_preds)

    avg_loss = total_loss / total_samples
    acc = accuracy_score(labels_np, preds_np)
    precision_fake = precision_score(labels_np, preds_np, pos_label=positive_class_idx, zero_division=0)
    recall_fake = recall_score(labels_np, preds_np, pos_label=positive_class_idx, zero_division=0)
    f1_fake = f1_score(labels_np, preds_np, pos_label=positive_class_idx, zero_division=0)
    cm = confusion_matrix(labels_np, preds_np)

    pred_df = pd.DataFrame(all_rows)
    pred_df.to_csv(save_csv_path, index=False, encoding="utf-8-sig")

    metrics = {
        "split": split_name,
        "loss": float(avg_loss),
        "accuracy": float(acc),
        "precision_fake": float(precision_fake),
        "recall_fake": float(recall_fake),
        "f1_fake": float(f1_fake),
        "confusion_matrix": cm.tolist(),
        "eval_time_sec": float(elapsed_sec),
        "eval_time_min": float(elapsed_min),
        "num_samples": int(total_samples),
    }

    print(f"\n[{split_name}]")
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
    print("Saved predictions:", save_csv_path)

    return metrics, pred_df


# ------------------------------------------------------------
# Run original and JPEG evaluation
# ------------------------------------------------------------

original_pred_path = eval_dir / "predictions_original.csv"
jpeg_pred_path = eval_dir / "predictions_jpeg_q30_controlled.csv"

original_metrics, original_pred_df = evaluate_and_save_predictions(
    model=model,
    loader=test_original_loader,
    criterion=criterion,
    device=device,
    class_names=class_names,
    positive_class_idx=positive_class_idx,
    save_csv_path=original_pred_path,
    split_name="original"
)

jpeg_metrics, jpeg_pred_df = evaluate_and_save_predictions(
    model=model,
    loader=test_jpeg_loader,
    criterion=criterion,
    device=device,
    class_names=class_names,
    positive_class_idx=positive_class_idx,
    save_csv_path=jpeg_pred_path,
    split_name="jpeg_q30_controlled"
)


# ------------------------------------------------------------
# Save metrics JSON
# ------------------------------------------------------------

metrics_dict = {
    "experiment_name": CONFIG["experiment_name"],
    "model_name": CONFIG["model_name"],
    "method_name": CONFIG["method_name"],
    "best_epoch": best_epoch,
    "best_val_f1": best_val_f1,
    "original": original_metrics,
    "jpeg_q30_controlled": jpeg_metrics,
}

metrics_json_path = eval_dir / "metrics_original_and_jpeg_q30_controlled.json"

with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_dict, f, indent=2, ensure_ascii=False)

print("\nSaved metrics JSON:", metrics_json_path)

Loading best checkpoint: /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_bg_suppression_patch_lambda0001_expand_10ep/best_checkpoint.pt
Best epoch: 5
Best val F1: 0.948768472906404
Original test size: 12005
JPEG test size: 12005
Original class_to_idx: {'fake': 0, 'real': 1}
JPEG class_to_idx: {'fake': 0, 'real': 1}


Eval original:   0%|          | 0/751 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (143040000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (121554000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(



[original]
{
  "split": "original",
  "loss": 0.21396128460873698,
  "accuracy": 0.9447730112453144,
  "precision_fake": 0.9358157765801078,
  "recall_fake": 0.955,
  "f1_fake": 0.945310566691413,
  "confusion_matrix": [
    [
      5730,
      270
    ],
    [
      393,
      5612
    ]
  ],
  "eval_time_sec": 4221.345234632492,
  "eval_time_min": 70.35575391054154,
  "num_samples": 12005
}
Saved predictions: /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_bg_suppression_patch_lambda0001_expand_10ep/eval_jpeg_q30_controlled/predictions_original.csv


Eval jpeg_q30_controlled:   0%|          | 0/751 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (143040000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (121554000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(



[jpeg_q30_controlled]
{
  "split": "jpeg_q30_controlled",
  "loss": 0.2352020616783515,
  "accuracy": 0.9386089129529362,
  "precision_fake": 0.9508309062874765,
  "recall_fake": 0.925,
  "f1_fake": 0.9377376024330489,
  "confusion_matrix": [
    [
      5550,
      450
    ],
    [
      287,
      5718
    ]
  ],
  "eval_time_sec": 6070.866014957428,
  "eval_time_min": 101.18110024929047,
  "num_samples": 12005
}
Saved predictions: /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_bg_suppression_patch_lambda0001_expand_10ep/eval_jpeg_q30_controlled/predictions_jpeg_q30_controlled.csv

Saved metrics JSON: /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_bg_suppression_patch_lambda0001_expand_10ep/eval_jpeg_q30_controlled/metrics_original_and_jpeg_q30_controlled.json


# **09_generate_summary_table**

In [13]:
# ============================================================
# 09_generate_summary_table
# Generate summary table for original vs JPEG q30 evaluation
# ============================================================

eval_dir = Path(CONFIG["output_dir"]) / "eval_jpeg_q30_controlled"

metrics_json_path = eval_dir / "metrics_original_and_jpeg_q30_controlled.json"

with open(metrics_json_path, "r", encoding="utf-8") as f:
    metrics_dict = json.load(f)

original = metrics_dict["original"]
jpeg = metrics_dict["jpeg_q30_controlled"]

summary_row = {
    "experiment_name": CONFIG["experiment_name"],
    "model_name": CONFIG["model_name"],
    "method_name": CONFIG["method_name"],
    "lambda_bg": CONFIG.get("lambda_bg", None),
    "best_epoch": metrics_dict.get("best_epoch", None),
    "best_val_f1": metrics_dict.get("best_val_f1", None),

    "original_accuracy": original["accuracy"],
    "original_precision_fake": original["precision_fake"],
    "original_recall_fake": original["recall_fake"],
    "original_f1_fake": original["f1_fake"],

    "jpeg_accuracy": jpeg["accuracy"],
    "jpeg_precision_fake": jpeg["precision_fake"],
    "jpeg_recall_fake": jpeg["recall_fake"],
    "jpeg_f1_fake": jpeg["f1_fake"],

    "accuracy_drop": original["accuracy"] - jpeg["accuracy"],
    "recall_fake_drop": original["recall_fake"] - jpeg["recall_fake"],
    "f1_fake_drop": original["f1_fake"] - jpeg["f1_fake"],

    "original_loss": original["loss"],
    "jpeg_loss": jpeg["loss"],

    "original_num_samples": original["num_samples"],
    "jpeg_num_samples": jpeg["num_samples"],

    "original_confusion_matrix": json.dumps(original["confusion_matrix"]),
    "jpeg_confusion_matrix": json.dumps(jpeg["confusion_matrix"]),

    "original_eval_time_min": original["eval_time_min"],
    "jpeg_eval_time_min": jpeg["eval_time_min"],
}

summary_df = pd.DataFrame([summary_row])

summary_path = eval_dir / "summary_jpeg_q30_controlled.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Saved summary:", summary_path)
display(summary_df)

Saved summary: /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_bg_suppression_patch_lambda0001_expand_10ep/eval_jpeg_q30_controlled/summary_jpeg_q30_controlled.csv


,experiment_name,model_name,method_name,lambda_bg,best_epoch,best_val_f1,original_accuracy,original_precision_fake,original_recall_fake,original_f1_fake,...,recall_fake_drop,f1_fake_drop,original_loss,jpeg_loss,original_num_samples,jpeg_num_samples,original_confusion_matrix,jpeg_confusion_matrix,original_eval_time_min,jpeg_eval_time_min
0,vit_b16_bg_suppression_patch_lambda0001_expand...,vit_b16,Patch-level Background Suppression,0.001,5,0.948768,0.944773,0.935816,0.955,0.945311,...,0.03,0.007573,0.213961,0.235202,12005,12005,"[[5730, 270], [393, 5612]]","[[5550, 450], [287, 5718]]",70.355754,101.1811
